# Hyperparameter Tuning

This notebook demonstrates three hyperparameter search strategies on the telco fraud dataset:

| # | Section | Strategy |
|---|---------|----------|
| 0 | Setup | Imports, data loading, preprocessing |
| 1 | Baseline | Default model performance (benchmark) |
| 2 | GridSearchCV | Exhaustive grid search |
| 3 | RandomizedSearchCV | Random sampling over the same grid |
| 4 | Optuna (Bayesian) | Efficient search with pruning |
| 5 | Comparison | Side-by-side strategy comparison |
| 6 | Early Stopping | Optuna `MedianPruner` demonstration |
| 7 | Best Model | Retrain and final evaluation |

**Metric:** ROC-AUC (consistent with the `online_payment_fraud_detection.ipynb` notebook).  
**Dataset:** 10% subsample of the telco fraud CSV to keep tuning tractable.

## 0. Setup

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier

from ml_pipeline.use_cases.fraud_detection import (
    engineer_telco_features,
    get_fraud_models,
    load_telco_fraud_data,
    suggest_lr_params,
    suggest_rf_params,
    suggest_xgb_params,
    TARGET_COL,
)
from ml_pipeline.core.preprocessing import split_features_target, train_test_split_data
from ml_pipeline.core.evaluation import evaluate_models
from ml_pipeline.core.tuning import (
    compare_search_strategies,
    plot_optimization_history,
    plot_param_importances,
    run_optuna_study,
    sklearn_cv_objective,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
CV_FOLDS = 3       # kept low for notebook speed
N_TRIALS = 30      # Optuna trials
N_ITER = 20        # RandomizedSearchCV iterations
SUBSAMPLE_FRAC = 0.10  # fraction of full dataset to use

print("Libraries imported successfully.")

In [ ]:
# Load and subsample the dataset
raw = load_telco_fraud_data()
raw = raw.sample(frac=SUBSAMPLE_FRAC, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Working with {len(raw):,} rows ({SUBSAMPLE_FRAC:.0%} subsample)")

# Feature engineering
df = engineer_telco_features(raw)

# Train / test split (stratified)
X, y = split_features_target(df, target_col=TARGET_COL)
X_train, X_test, y_train, y_test = train_test_split_data(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=True
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {X_train.shape[1]}")
print(f"Fraud rate (train): {y_train.mean():.3%}")

## 1. Baseline — Default Hyperparameters

Evaluate all three default models first to establish the benchmark.

In [ ]:
baseline_models = get_fraud_models()

print("Training baseline models...")
for model in baseline_models:
    model.fit(X_train, y_train)

baseline_results = evaluate_models(baseline_models, X_train, y_train, X_test, y_test)

In [ ]:
baseline_df = pd.DataFrame([
    {"Model": r["name"], "Train AUC": r["train_auc"], "Test AUC": r["test_auc"]}
    for r in baseline_results
])
print("Baseline results:")
print(baseline_df.to_string(index=False))

# We focus on XGBoost as it typically performs best
xgb_baseline_auc = baseline_df.loc[
    baseline_df["Model"] == "XGBClassifier", "Test AUC"
].values[0]
print(f"\nXGBoost baseline test AUC: {xgb_baseline_auc:.4f}  ← target to beat")

## 2. GridSearchCV

Exhaustive search over a small XGBoost parameter grid.

In [ ]:
from sklearn.model_selection import GridSearchCV

xgb_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6],
    "learning_rate": [0.05, 0.1],
}

print(f"Grid size: {2**3} = 8 combinations × {CV_FOLDS} folds = {8*CV_FOLDS} fits")

t0 = time.perf_counter()
grid_search = GridSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
    xgb_param_grid,
    cv=CV_FOLDS,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)
grid_time = time.perf_counter() - t0

print(f"\nBest params : {grid_search.best_params_}")
print(f"Best CV AUC : {grid_search.best_score_:.4f}")
print(f"Time        : {grid_time:.1f} s")

In [ ]:
# Evaluate the best GridSearch model on the held-out test set
grid_test_auc = roc_auc_score(y_test, grid_search.best_estimator_.predict_proba(X_test)[:, 1])
print(f"GridSearchCV best model — test AUC: {grid_test_auc:.4f}")

## 3. RandomizedSearchCV

Random sampling from continuous distributions — more efficient than grid search for larger spaces.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint, uniform

xgb_param_distributions = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(3, 9),
    "learning_rate": loguniform(0.01, 0.3),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "reg_lambda": loguniform(0.01, 10),
}

print(f"Running {N_ITER} random iterations × {CV_FOLDS} folds = {N_ITER*CV_FOLDS} fits")

t0 = time.perf_counter()
rand_search = RandomizedSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
    xgb_param_distributions,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)
rand_search.fit(X_train, y_train)
rand_time = time.perf_counter() - t0

print(f"\nBest params : {rand_search.best_params_}")
print(f"Best CV AUC : {rand_search.best_score_:.4f}")
print(f"Time        : {rand_time:.1f} s")

In [ ]:
rand_test_auc = roc_auc_score(y_test, rand_search.best_estimator_.predict_proba(X_test)[:, 1])
print(f"RandomizedSearchCV best model — test AUC: {rand_test_auc:.4f}")

## 4. Optuna — Bayesian Optimisation

Optuna uses a Tree-structured Parzen Estimator (TPE) to focus search on promising regions of the parameter space, spending fewer evaluations on clearly bad configurations.

In [ ]:
def xgb_objective(trial):
    """Optuna objective — proposes XGBoost params, returns mean CV ROC-AUC."""
    params = suggest_xgb_params(trial)
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring="roc_auc", n_jobs=-1)
    return scores.mean()


print(f"Running Optuna study: {N_TRIALS} trials, {CV_FOLDS}-fold CV...")
t0 = time.perf_counter()
xgb_study = run_optuna_study(
    xgb_objective,
    n_trials=N_TRIALS,
    direction="maximize",
    study_name="xgb-fraud",
    use_pruner=False,  # pruner shown separately in Section 6
)
optuna_time = time.perf_counter() - t0

print(f"\nBest params : {xgb_study.best_params}")
print(f"Best CV AUC : {xgb_study.best_value:.4f}")
print(f"Time        : {optuna_time:.1f} s")

In [ ]:
# Evaluate Optuna best model on the held-out test set
optuna_best_model = XGBClassifier(**xgb_study.best_params)
optuna_best_model.fit(X_train, y_train)
optuna_test_auc = roc_auc_score(y_test, optuna_best_model.predict_proba(X_test)[:, 1])
print(f"Optuna best model — test AUC: {optuna_test_auc:.4f}")

In [ ]:
# Visualise the optimization landscape
plot_optimization_history(xgb_study)

In [ ]:
plot_param_importances(xgb_study)

## 5. Strategy Comparison

Side-by-side summary across all three approaches.

In [ ]:
comparison = pd.DataFrame([
    {
        "Strategy": "Baseline (default)",
        "CV / Train AUC": xgb_baseline_auc,
        "Test AUC": xgb_baseline_auc,
        "Search time (s)": 0.0,
        "# Evaluations": 1,
    },
    {
        "Strategy": "GridSearchCV",
        "CV / Train AUC": grid_search.best_score_,
        "Test AUC": grid_test_auc,
        "Search time (s)": round(grid_time, 1),
        "# Evaluations": len(grid_search.cv_results_["mean_test_score"]),
    },
    {
        "Strategy": "RandomizedSearchCV",
        "CV / Train AUC": rand_search.best_score_,
        "Test AUC": rand_test_auc,
        "Search time (s)": round(rand_time, 1),
        "# Evaluations": N_ITER,
    },
    {
        "Strategy": "Optuna (Bayesian)",
        "CV / Train AUC": xgb_study.best_value,
        "Test AUC": optuna_test_auc,
        "Search time (s)": round(optuna_time, 1),
        "# Evaluations": N_TRIALS,
    },
])

print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

strategies = comparison["Strategy"]
colors = ["#9b9b9b", "#4c72b0", "#dd8452", "#55a868"]

axes[0].bar(strategies, comparison["Test AUC"], color=colors)
axes[0].set_ylim(comparison["Test AUC"].min() - 0.005, 1.0)
axes[0].set_ylabel("Test ROC-AUC")
axes[0].set_title("Model Quality")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(strategies, comparison["Search time (s)"], color=colors)
axes[1].set_ylabel("Wall-clock time (s)")
axes[1].set_title("Search Time")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Hyperparameter Search Strategy Comparison", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Early Stopping with Optuna MedianPruner

The `MedianPruner` prunes a trial when its intermediate value falls below the median of completed trials at the same step.

Here we demonstrate it by reporting per-fold CV scores via `trial.report()` — if the running mean after each fold looks poor compared to completed trials, Optuna raises `TrialPruned` and skips the remaining folds.  This cuts wasted compute without any framework-specific callbacks.

In [ ]:
def xgb_pruning_objective(trial):
    """
    XGBoost objective with per-fold intermediate value reporting.

    After each CV fold the running mean AUC is reported to Optuna.
    The MedianPruner can stop unpromising trials before all folds complete.
    """
    params = suggest_xgb_params(trial)
    model = XGBClassifier(**params)

    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_fold_train = X_train.iloc[train_idx]
        y_fold_train = y_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_val = y_train.iloc[val_idx]

        model.fit(X_fold_train, y_fold_train)
        fold_auc = roc_auc_score(y_fold_val, model.predict_proba(X_fold_val)[:, 1])
        fold_scores.append(fold_auc)

        # Report intermediate value so MedianPruner can evaluate this trial
        trial.report(float(np.mean(fold_scores)), step)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))


print(f"Running pruning study: {N_TRIALS} trials with MedianPruner...")
t0 = time.perf_counter()
pruning_study = run_optuna_study(
    xgb_pruning_objective,
    n_trials=N_TRIALS,
    direction="maximize",
    study_name="xgb-fraud-pruning",
    use_pruner=True,
)
pruning_time = time.perf_counter() - t0

completed = [t for t in pruning_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in pruning_study.trials if t.state == optuna.trial.TrialState.PRUNED]

print(f"\nCompleted trials : {len(completed)}")
print(f"Pruned trials    : {len(pruned)}")
print(f"Best CV AUC      : {pruning_study.best_value:.4f}")
print(f"Time             : {pruning_time:.1f} s")

In [ ]:
# Visual comparison of trial outcomes
trial_states = pd.Series(
    [t.state.name for t in pruning_study.trials]
).value_counts()

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(
    trial_states.index,
    trial_states.values,
    color=["#55a868", "#dd8452", "#c44e52"][:len(trial_states)],
)
ax.set_ylabel("Count")
ax.set_title("Trial outcomes with MedianPruner")
plt.tight_layout()
plt.show()

if len(pruned) > 0:
    print(f"Pruning saved ~{len(pruned) / N_TRIALS:.0%} of potential compute.")
else:
    print("No trials pruned (MedianPruner needs warm-up trials; increase N_TRIALS to see pruning in action.)")

## 7. Best Model — Final Evaluation

Retrain the best XGBoost configuration found by Optuna on the **full training set**, then evaluate on the held-out test set.

In [ ]:
print("Best hyperparameters found by Optuna:")
for k, v in xgb_study.best_params.items():
    print(f"  {k}: {v}")

best_xgb = XGBClassifier(**xgb_study.best_params)
best_xgb.fit(X_train, y_train)

final_test_auc = roc_auc_score(y_test, best_xgb.predict_proba(X_test)[:, 1])
print(f"\nFinal model — test ROC-AUC: {final_test_auc:.4f}")
print(f"Baseline XGBoost AUC      : {xgb_baseline_auc:.4f}")
lift = final_test_auc - xgb_baseline_auc
sign = "+" if lift >= 0 else ""
print(f"Lift                      : {sign}{lift:.4f}")

In [ ]:
from ml_pipeline.core.evaluation import plot_confusion_matrix

plot_confusion_matrix(best_xgb, X_test, y_test)

## Summary

| Strategy | Strength | Weakness |
|---|---|---|
| **GridSearchCV** | Deterministic, exhaustive | Exponential cost; impractical for large grids |
| **RandomizedSearchCV** | Same budget, wider coverage | No learning from previous trials |
| **Optuna (TPE)** | Learns from history; efficient | Stochastic; needs warm-up trials |
| **Optuna + Pruning** | Cuts unpromising trials early | Requires per-step reporting |

For production workloads:
- Prefer **Optuna** when you have a large parameter space or limited compute budget.
- Use **GridSearchCV** only for small, well-understood grids where exhaustiveness matters.
- Enable **pruning** when your estimator supports incremental fitting or per-step evaluation.